# Ollama Model Benchmark
Benchmarks every locally installed Ollama model on:
- **load_time_s** — seconds from cold start to first token
- **response_time_s** — seconds from first token to last token
- **total_time_s** — end-to-end wall time

Each model is unloaded before its run to ensure a cold start.

In [1]:
import time
import ollama
import pandas as pd

_models_df = pd.read_csv("model_tags_clean.csv")

models = []
for m in ollama.list().models:
    models.append(m.model)

_models_df = _models_df[_models_df["tag"].isin(models)].sort_values("size")
models = _models_df["tag"].tolist()
_models_df

,model,tag,size,updated,Image,Text,context
1635,gemma3,gemma3:1b,0.795898,365,0,1,32000
1540,gemma2,gemma2:2b,1.600000,365,0,1,8000
2800,llama3.2,llama3.2:3b-instruct-q5_K_M,2.300000,365,0,1,128000
2957,llava-phi3,llava-phi3:3.8b-mini-q4_0,2.900000,365,1,1,4000
1636,gemma3,gemma3:4b,3.300000,365,1,1,128000
2674,llama3.1,llama3.1:8b-instruct-q4_K_M,4.900000,365,0,1,128000
3157,mistral,mistral:7b-instruct-v0.3-q5_K_S,5.000000,300,0,1,32000
1665,gemma3n,gemma3n:e2b-it-q4_K_M,5.600000,270,0,1,32000
3239,mistral-nemo,mistral-nemo:12b-instruct-2407-q4_K_M,7.500000,240,0,1,1000000
2822,llama3.2-vision,llama3.2-vision:11b-instruct-q4_K_M,7.800000,330,1,1,128000


In [2]:
_models_str = _models_df.to_string(index=False)
PROMPTS = {
    "short": "In one sentence, what is the capital of France?",
    "long_in": (
        f"Here is a table of Ollama model metadata:\n\n{_models_str}\n\n"
        "Based only on the data above, which model is the largest? "
        "Answer with just the model name, nothing else."
    ),
}

In [3]:
N = 3

OPTIONS = {
    "temperature": 0.7,
    "num_predict": 512,
    "num_ctx": 4096,
    "thinking": False,
}


def unload_model(model: str) -> None:
    ollama.generate(model=model, prompt="", keep_alive=0, options=OPTIONS)


def time_generate(model: str, prompt: str) -> tuple[float, str]:
    t = time.perf_counter()
    resp = ollama.generate(model=model, prompt=prompt, options=OPTIONS)
    return round(time.perf_counter() - t, 3), resp.response

In [4]:
results = []

for i, model in enumerate(models, 1):
    print(f"[{i}/{len(models)}] {model}")
    try:
        for run in range(1, N + 1):
            row = {"model": model, "run": run}

            if run == 1:
                row["load_time_s"], _ = time_generate(model, "")
                print(f"  run {run}  load {row['load_time_s']:.2f}s")
            else:
                row["load_time_s"] = None
                print(f"  run {run}")

            for prompt_id, prompt_text in PROMPTS.items():
                elapsed, text = time_generate(model, prompt_text)
                row[f"{prompt_id}_time_s"] = elapsed
                row[f"{prompt_id}_text"] = text
                print(f"    {prompt_id:<6}  {elapsed:.2f}s")

            results.append(row)

    except Exception as e:
        print(f"  ERROR: {e}")
        row.setdefault("load_time_s", None)
        for prompt_id in PROMPTS:
            row.setdefault(f"{prompt_id}_time_s", None)
            row.setdefault(f"{prompt_id}_text", None)
        results.append(row)
    finally:
        unload_model(model)

print("\nDone.")

[1/12] gemma3:1b
  run 1  load 1.98s
    short   0.38s
    long_in  0.48s
  run 2
    short   0.33s
    long_in  0.47s
  run 3
    short   0.32s
    long_in  0.48s
[2/12] gemma2:2b
  run 1  load 3.22s
    short   0.32s
    long_in  0.67s
  run 2
    short   0.31s
    long_in  0.67s
  run 3
    short   0.31s
    long_in  0.67s
[3/12] llama3.2:3b-instruct-q5_K_M
  run 1  load 3.97s
    short   0.33s
    long_in  0.92s
  run 2
    short   0.28s
    long_in  0.90s
  run 3
    short   0.28s
    long_in  0.89s
[4/12] llava-phi3:3.8b-mini-q4_0
  run 1  load 4.73s
    short   0.20s
    long_in  2.58s
  run 2
    short   0.18s
    long_in  2.87s
  run 3
    short   0.18s
    long_in  2.87s
[5/12] gemma3:4b
  run 1  load 4.31s
    short   0.50s
    long_in  1.96s
  run 2
    short   0.45s
    long_in  1.96s
  run 3
    short   0.44s
    long_in  1.96s
[6/12] llama3.1:8b-instruct-q4_K_M
  run 1  load 6.25s
    short   0.41s
    long_in  1.69s
  run 2
    short   0.40s
    long_in  1.69s
  run 3
 

In [7]:
cols = ["model", "run", "load_time_s", "short_time_s", "short_text", "long_in_time_s", "long_in_text"]
df = pd.DataFrame(results)[cols]
(
df.sort_values("long_in_time_s")
.style.background_gradient(
    subset=["load_time_s", "short_time_s", "long_in_time_s"],
    cmap="RdYlGn_r")
)

,model,run,load_time_s,short_time_s,short_text,long_in_time_s,long_in_text
1,gemma3:1b,2,nan,0.331000,The capital of France is Paris.,0.474000,gemma3
2,gemma3:1b,3,nan,0.324000,The capital of France is Paris.,0.477000,gemma3
0,gemma3:1b,1,1.980000,0.381000,The capital of France is Paris.,0.483000,gemma3
3,gemma2:2b,1,3.222000,0.323000,The capital of France is Paris. 🇫🇷,0.669000,gpt-oss
4,gemma2:2b,2,nan,0.313000,The capital of France is Paris. 🇫🇷,0.670000,gpt-oss
5,gemma2:2b,3,nan,0.314000,The capital of France is Paris. 🇫🇷,0.674000,gpt-oss
8,llama3.2:3b-instruct-q5_K_M,3,nan,0.283000,The capital of France is Paris.,0.885000,mistral-nemo
7,llama3.2:3b-instruct-q5_K_M,2,nan,0.282000,The capital of France is Paris.,0.897000,llava-phi3
6,llama3.2:3b-instruct-q5_K_M,1,3.969000,0.332000,The capital of France is Paris.,0.922000,llama3.2-vision
15,llama3.1:8b-instruct-q4_K_M,1,6.245000,0.413000,The capital of France is Paris.,1.685000,gpt-oss


In [6]:
print(df.to_string())

                                    model  run  load_time_s  short_time_s                             short_text  long_in_time_s                                                                                    long_in_text
0                               gemma3:1b    1        1.980         0.381        The capital of France is Paris.           0.483                                                                                        gemma3\n
1                               gemma3:1b    2          NaN         0.331        The capital of France is Paris.           0.474                                                                                        gemma3\n
2                               gemma3:1b    3          NaN         0.324        The capital of France is Paris.           0.477                                                                                        gemma3\n
3                               gemma2:2b    1        3.222         0.323  The capital of France is 